In [17]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import pandas as pd
from src.parsers import parse_stories


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
df=pd.read_csv("../data/interim/listings_parsed.csv")

In [19]:
df.shape

(183490, 22)

In [20]:
df.head()

,Listing Title,Area,Distances to Railway Stations,Building Age,Stories Tall,Floor,Price,Administrative Costs,Deposit,Gratuity,...,Links to Listing,jnc,Price_yen,Administrative Costs_yen,Deposit_yen,Gratuity_yen,Admin_yen,deposit_months,gratuity_months,lookalike_id
0,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000055105229/?bc=...,55105229,65000,3000.0,NaN,NaN,3000.0,NaN,NaN,72960
1,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.5万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000055105230/?bc=...,55105230,65000,3000.0,NaN,NaN,3000.0,NaN,NaN,72960
2,ルアナ,東京都あきる野市伊奈,"ＪＲ五日市線/武蔵増戸駅 歩4分,ＪＲ五日市線/武蔵引田駅 歩24分",新築,2階建,1階,6.8万円,3000円,-,-,...,https://suumo.jp/chintai/jnc_000054648287/?bc=...,54648287,68000,3000.0,NaN,NaN,3000.0,NaN,NaN,72961
3,カーザ コリーナ,東京都青梅市野上町２,ＪＲ青梅線/河辺駅 歩15分,築5年,3階建,3階,8.5万円,4000円,8.5万円,4.25万円,...,https://suumo.jp/chintai/jnc_000055414464/?bc=...,55414464,85000,4000.0,85000.0,42500.0,4000.0,1.0,0.5,20237
4,京王相模原線 京王よみうりランド駅 2階建 築14年,東京都稲城市矢野口,"京王相模原線/京王よみうりランド駅 歩10分,ＪＲ南武線/矢野口駅 歩17分,京王相模原線/...",築14年,2階建,1階,8.1万円,3900円,-,-,...,https://suumo.jp/chintai/jnc_000037983769/?bc=...,37983769,81000,3900.0,NaN,NaN,3900.0,NaN,NaN,84797


In [21]:
df.columns

Index(['Listing Title', 'Area', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id'],
      dtype='str')

In [22]:
df[["stories_above", "stories_below"]] = df["Stories Tall"].apply(lambda t: pd.Series(parse_stories(t)))
print((df["stories_above"] == "FAILED").sum())

213


In [23]:
df[df["Stories Tall"] == "平屋"][["Price_yen", "Square Meters", "Rooms"]].describe()

,Price_yen
count,213.000000
mean,96704.225352
std,56725.687428
min,39000.000000
25%,65000.000000
50%,80000.000000
75%,100000.000000
max,350000.000000


In [24]:
df.loc[df["stories_above"] == "FAILED", "Stories Tall"].value_counts().head(10)

Stories Tall
平屋    213
Name: count, dtype: int64

In [27]:
df['Floor'].value_counts().head(50) 

Floor
1階       60974
2階       54627
3階       26766
4階       12926
5階        7333
6階        4580
7階        3177
8階        2439
1-2階      2054
9階        1765
10階       1546
-          997
11階        857
B1階        479
13階        454
12階        418
1-3階       270
14階        258
15階        180
21階        143
19階        105
2-3階       104
16階         88
23階         74
24階         64
30階         61
18階         55
25階         46
29階         46
B2階         45
3-4階        42
17階         42
20階         41
35階         38
1-1階        38
B1-1階       36
22階         28
4-5階        24
2-1階        21
28階         17
32階         17
B1-2階       16
51階         16
33階         14
1-階         12
2-0階        11
27階          8
1-4階         8
1-10階        8
34階          7
Name: count, dtype: int64

In [28]:
df["Floor"].str.replace(r"\d", "9", regex=True).value_counts()

Floor
9階        174587
99階         4652
9-9階        2617
-            997
B9階          529
B9-9階         55
9-階           27
9-99階          9
M9階            5
99-階           4
99-99階         3
99-9階          2
9B9階           2
B9-階           1
Name: count, dtype: int64

In [29]:
from src.parsers import parse_floor
for t in ["1階", "12階", "1-2階", "1-階", "B1階", "B1-1階", "-", "M1階", "1B1階"]:
    print(repr(t), parse_floor(t))

'1階' (1, 1, 0)
'12階' (12, 12, 0)
'1-2階' (1, 2, 1)
'1-階' (1, 1, 1)
'B1階' (-1, -1, 0)
'B1-1階' (-1, 1, 1)
'-' (None, None, None)
'M1階' ('FAILED', None, None)
'1B1階' ('FAILED', None, None)


In [30]:
df[["floor_min", "floor_max", "is_multi_floor"]] = df["Floor"].apply(lambda t: pd.Series(parse_floor(t)))
print((df["floor_min"] == "FAILED").sum())

7


In [31]:
df.loc[df["floor_min"] == "FAILED", "Floor"].value_counts()

Floor
M2階     5
1B2階    2
Name: count, dtype: int64

In [32]:
for c in ["floor_min", "floor_max", "is_multi_floor"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df["is_multi_floor"].value_counts())
print(df[["floor_min", "floor_max"]].describe())
print(df["floor_min"].isna().sum())

is_multi_floor
0.0    179768
1.0      2718
Name: count, dtype: int64
           floor_min      floor_max
count  182486.000000  182486.000000
mean        2.690672       2.706525
std         2.643514       2.637635
min        -4.000000      -4.000000
25%         1.000000       1.000000
50%         2.000000       2.000000
75%         3.000000       3.000000
max        51.000000      51.000000
1004
